# Core variant set scoring

Generate prediction probabilities for all variants in the imputed core feature matrix, using the trained FuncVEP and ClinVEP models while excluding variants used for model training. These scores are used for downstream benchmarking; the full inference workflow including imputation for new variants is provided in notebook 06_inference_on_new_variants.ipynb.

In [1]:
import os
import numpy as np
import pandas as pd
import joblib

feature_matrix = pd.read_parquet("../data/intermediate/feature_matrix_imputed.parquet")
feature_matrix.columns = feature_matrix.columns.str.replace(" ", "_")

id_column = "ID"
ensg_column = "ensg"

feature_matrix[id_column] = feature_matrix[id_column].astype(str)
if ensg_column in feature_matrix.columns:
    feature_matrix[ensg_column] = feature_matrix[ensg_column].astype(str)

model_names = [
    "FuncVEP_CTI",
    "FuncVEP_CTE",
    "FuncVEP_SP",
    "ClinVEP_CTI",
    "ClinVEP_CTE",
    "ClinVEP_SP",
]

scores = feature_matrix[[id_column, ensg_column]].copy()

for model_name in model_names:
    model_dir = f"../models/{model_name}"
    os.makedirs(model_dir, exist_ok=True)

    lgb_model = joblib.load(os.path.join(model_dir, "model.pkl"))
    trained_features = list(lgb_model.feature_name_)

    trained_on = pd.read_csv(
        os.path.join(model_dir, "training_set.txt"),
        sep="\t",
    )

    trained_on[id_column] = trained_on[id_column].astype(str)
    trained_on[ensg_column] = trained_on[ensg_column].astype(str)

    all_idx = pd.MultiIndex.from_frame(
        feature_matrix[[id_column, ensg_column]]
    )
    train_idx = pd.MultiIndex.from_frame(
        trained_on[[id_column, ensg_column]]
    )
    keep_mask = ~all_idx.isin(train_idx)

    missing_features = set(trained_features) - set(feature_matrix.columns)
    if missing_features:
        print(f"WARNING: missing features for {model_name}:")
        print(missing_features)

    X = feature_matrix.loc[keep_mask, trained_features]
    X = X.apply(pd.to_numeric, errors="coerce")

    y_pred_proba = lgb_model.predict_proba(X)[:, 1]

    model_scores = np.full(shape=len(feature_matrix), fill_value=np.nan, dtype=float)
    model_scores[keep_mask] = y_pred_proba

    scores[model_name] = model_scores

del feature_matrix

result_dir = "../results/predictions"
os.makedirs(result_dir, exist_ok=True)

out_path = os.path.join(result_dir, "core_variant_set_scores.txt")
scores.to_csv(out_path, sep="\t", index=False)

print("Saved core variant scores to:", out_path)

scores

[LightGBM] [Warning] lambda_l2 is set=2.514857127963241, reg_lambda=0.0 will be ignored. Current value: lambda_l2=2.514857127963241
[LightGBM] [Warning] lambda_l1 is set=0.023591448340076825, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.023591448340076825
[LightGBM] [Warning] lambda_l2 is set=2.518779208282009, reg_lambda=0.0 will be ignored. Current value: lambda_l2=2.518779208282009
[LightGBM] [Warning] lambda_l1 is set=0.01606167254053278, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.01606167254053278
[LightGBM] [Warning] lambda_l2 is set=0.1972128852356803, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1972128852356803
[LightGBM] [Warning] lambda_l1 is set=0.028033340732676933, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.028033340732676933
[LightGBM] [Warning] lambda_l2 is set=1.0376445056377905, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.0376445056377905
[LightGBM] [Warning] lambda_l1 is set=0.007983675628043422,

,ID,ensg,FuncVEP_CTI,FuncVEP_CTE,FuncVEP_SP,ClinVEP_CTI,ClinVEP_CTE,ClinVEP_SP
0,10-100057090-C-T,ENSG00000120054,0.005382,0.004680,0.033497,0.000792,0.003038,0.016552
1,10-100069757-C-T,ENSG00000120054,0.389112,0.257733,0.128476,0.116737,0.136158,0.058580
2,10-100076062-G-A,ENSG00000120054,0.003516,0.010462,0.086783,0.001022,0.058010,0.039352
3,10-100081405-G-A,ENSG00000120054,0.003943,0.001977,0.058972,0.000658,0.006199,0.031094
4,10-100152307-T-C,ENSG00000107566,0.000863,0.000657,0.024006,0.000583,0.001411,0.003902
...,...,...,...,...,...,...,...,...
1117936,X-154442668-A-C,ENSG00000203879,0.196032,0.377131,0.166843,0.353600,0.068080,0.039384
1117937,X-154442668-A-G,ENSG00000203879,0.256512,0.326721,0.222517,0.369332,0.095153,0.109578
1117938,X-154442668-A-T,ENSG00000203879,0.259031,0.390279,0.162771,0.370953,0.119307,0.085728
1117939,X-154442669-G-C,ENSG00000203879,0.192984,0.281070,0.154832,0.111842,0.049268,0.051070
